In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import Model, layers,models


In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Reshape the data for CNN input 
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

# CNNs
## Simple Convolutional Neural Networks are usually made up of a few layers:
1. Convolutional Layer
2. Activation Layer
3. Pooling Layer
4. Fully Connected Layer
5. Output Layer


## Minimal
- **Convolutional Layer**: At least one Conv2D layer to extract features
    - Learns spatial patterns in the input
    - Defined by kernel size, number of filters, stride, and padding
- **Activation Function**: Typically ReLU
    - Introduces non-linearity
    - Helps learn complex patterns
- **Pooling Layer**: Usually MaxPool2D
    - Reduces spatial dimensions
    - Makes the model more robust to position variations
- **Fully Connected Layer**: For final classification
    - Converts features to class probabilities
    - Output size matches number of classes


In [ ]:

class MinimalCNN(Model):
    def __init__(self, num_classes):
        super(MinimalCNN, self).__init__()
        
        # Minimum required layers
        self.conv1 = layers.Conv2D(
            filters=16,           # Number of filters
            kernel_size=(3, 3),   # 3x3 filter
            strides=1,
            padding='same', #
            activation='relu',    # Including activation in the Conv layer
            input_shape=(28, 28, 1)  # For grayscale images like MNIST
        )
        
        self.pool = layers.MaxPooling2D(pool_size=(2, 2))
        self.flatten = layers.Flatten()
        self.fc = layers.Dense(num_classes)
        
    def call(self, x):
        x = self.conv1(x)    # Conv2D + ReLU
        x = self.pool(x)     # Pooling
        x = self.flatten(x)  # Flatten for Dense layer
        x = self.fc(x)       # Classification layer
        return x


# Example usage:
model = MinimalCNN(num_classes=10)  # For class-based approach

# Compile and train

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:

def create_simple_cnn(input_shape, num_classes):
    model = models.Sequential([
        # Convolutional Layer
        layers.Conv2D(1, (3, 3), padding='same', input_shape=input_shape),
        # Activation Layer
        layers.Activation('relu'),
        # Pooling Layer
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        # Another set of Conv, Activation, and Pooling layers
        
        
        # Flatten the output for the fully connected layer
        layers.Flatten(),
        
        # Fully Connected Layer
        layers.Dense(8),
        layers.Activation('relu'),
        
        # Output Layer
        layers.Dense(num_classes),
        layers.Activation('softmax')
    ])
    
    return model

# Example usage
input_shape = (28, 28, 1)  # for MNIST dataset
num_classes = 10  # 10 digits

simple_model = create_simple_cnn(input_shape, num_classes)
simple_model.summary()


simple_model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])




In [ ]:
simple_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))
test_loss, test_acc = simple_model.evaluate(x_test, y_test, verbose=2)
print(f'\nTest accuracy: {test_acc}')

In [ ]:
simple_model.save('simple_model.keras')


In [ ]:
predictions = simple_model.predict(x_test)

def plot_image(i, predictions_array, true_label, img):
    predictions_array, true_label, img = predictions_array[i], true_label[i], img[i]
    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    
    plt.imshow(img.reshape(28, 28), cmap=plt.cm.binary)

    predicted_label = np.argmax(predictions_array)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'
    
    plt.xlabel(f"{predicted_label} ({100*np.max(predictions_array):2.0f}%) (True: {true_label})", color=color)

num_rows = 5
num_cols = 3
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
    plt.subplot(num_rows, 2*num_cols, 2*i+1)
    plot_image(i, predictions, y_test, x_test)
plt.tight_layout()
plt.show()

In [ ]:
#Go wild and experiment and see what happens

def create_custom_cnn(input_shape, num_classes):
    model = models.Sequential([
            # Convolutional Layer with 32 filters, 3x3 kernel size, 'same' padding to maintain output dimensions,
            # and strides set to (1, 1) (default), meaning the filter moves one pixel at a time.
            # To experiment, change strides to (2, 2) to see how it impacts the output dimensions.
            layers.Conv2D(32, (3, 3), strides=(1, 1), padding='same', input_shape=input_shape),
            
            # Activation Layer with ReLU activation. Alternatives: 'sigmoid', 'tanh', 'softmax' (for output layer), or 'leaky_relu' (variant of ReLU).
            layers.Activation('relu'),
            
            # Pooling Layer with 2x2 pooling window and default stride of 2 (reduces spatial size by half).
            # Options: MaxPooling2D (common for feature extraction) or AveragePooling2D (smoother, averages values).
            layers.MaxPooling2D(pool_size=(2, 2)),
            
            # Another Convolutional Layer with 64 filters, 3x3 kernel size, and 'same' padding to maintain dimensions.
            layers.Conv2D(64, (3, 3), padding='same'),
            
            # Activation Layer with ReLU.
            layers.Activation('relu'),
            
            # Pooling Layer with 2x2 window. Max pooling is often preferred for capturing strong features.
            layers.MaxPooling2D(pool_size=(2, 2)),
            
            # Flattening layer to convert 2D feature maps into a 1D vector for fully connected layers.
            layers.Flatten(),
            
            # Fully Connected Layer with 128 units. Can adjust units based on model complexity.
            layers.Dense(128),
            
            # Activation Layer with ReLU for non-linearity. Alternatives for fully connected layers: 'relu', 'tanh', or 'elu'.
            layers.Activation('relu'),
            
            # Output Layer with a number of units equal to the number of classes, softmax activation to get probabilities for each class.
            layers.Dense(num_classes),
            layers.Activation('softmax') # softmax for multi-class classification, can use sigmoid for binary classification.
        ])
        
    return model

In [ ]:
input_shape = (28, 28, 1)  
num_classes = 10 

custom_model = create_custom_cnn(input_shape, num_classes)
custom_model.summary()
custom_model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:

custom_model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))
test_loss, test_acc = simple_model.evaluate(x_test, y_test, verbose=2)
print(f'\nTest accuracy: {test_acc}')

In [ ]:
custom_model.save('custom_model.keras')
